# Notebook 10 — Model Training: XGBoost

## AI-Supply-Chain-Digital-Marketing

### Objective
Train the **XGBoost candidate selected by Notebook 09** using the training period only.

Notebook 09 completed the final four-model comparison:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

The selected candidate is **XGBoost based on validation F1**.

### Data protocol

- Training set: used for fitting XGBoost.
- Validation set: used only for a prediction sanity check.
- Test set: loaded only for integrity checks.
- Test data is **not used for training**.
- Test data is **not used for model selection**.
- Test predictions are **not generated**.
- Hyperparameter tuning is reserved for Notebook 11.
- Final test evaluation is reserved for Notebook 12.

This notebook deliberately fails if the Notebook 09 selection artifact does not say `XGBoost`. This prevents an older Logistic Regression selection artifact from being used accidentally.


In [1]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBClassifier


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "supply_chain"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR.resolve())
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Model directory exists:", MODEL_DIR.exists())


Base directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Processed directory exists: True
Model directory exists: True


## Cell 3 — Load preprocessed train, validation, and test data

In [2]:
X_train = pd.read_csv(
    PROCESSED_DIR / "X_train_preprocessed.csv"
)
y_train = pd.read_csv(
    PROCESSED_DIR / "y_train_preprocessed.csv"
).squeeze("columns")

X_validation = pd.read_csv(
    PROCESSED_DIR / "X_validation_preprocessed.csv"
)
y_validation = pd.read_csv(
    PROCESSED_DIR / "y_validation_preprocessed.csv"
).squeeze("columns")

# Test is loaded only for integrity checks.
# It is NOT used for fitting or prediction.
X_test = pd.read_csv(
    PROCESSED_DIR / "X_test_preprocessed.csv"
)
y_test = pd.read_csv(
    PROCESSED_DIR / "y_test_preprocessed.csv"
).squeeze("columns")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (3500, 590)
y_train: (3500,)
X_validation: (750, 590)
y_validation: (750,)
X_test: (750, 590)
y_test: (750,)


## Cell 4 — Validate input data

In [3]:
assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert len(y_train) == 3500
assert len(y_validation) == 750
assert len(y_test) == 750

assert list(X_train.columns) == list(X_validation.columns)
assert list(X_train.columns) == list(X_test.columns)

assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()
assert not X_test.isna().any().any()

assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_validation.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

assert set(pd.Series(y_train).unique()).issubset({0, 1})
assert set(pd.Series(y_validation).unique()).issubset({0, 1})
assert set(pd.Series(y_test).unique()).issubset({0, 1})

print("Input validation: PASS")
print("Training rows:", X_train.shape[0])
print("Training features:", X_train.shape[1])


Input validation: PASS
Training rows: 3500
Training features: 590


## Cell 5 — Read Notebook 09 final model-selection artifact

Only the final four-model selection artifact is accepted.

Expected file:

`models/supply_chain/all_four_model_selection_info.json`

Expected selected candidate:

`XGBoost`

If the file contains another model, this notebook stops instead of silently training the wrong candidate.


In [4]:
selection_path = MODEL_DIR / "all_four_model_selection_info.json"

if not selection_path.exists():
    raise FileNotFoundError(
        f"Notebook 09 selection artifact not found: {selection_path}\n"
        "Run Notebook 09 first."
    )

with open(selection_path, "r", encoding="utf-8") as f:
    selection_info = json.load(f)

selected_model_name = selection_info["selected_model"]
selection_metric = selection_info["selection_metric"]
selection_split = selection_info["selection_split"]

expected_models = {
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "LightGBM"
}

models_compared = set(selection_info["models_compared"])

assert models_compared == expected_models
assert selection_metric == "F1"
assert selection_split == "validation"

# Critical protection:
# Notebook 09 selected XGBoost. Do not allow an older artifact
# to make Notebook 10 train Logistic Regression.
assert selected_model_name == "XGBoost", (
    f"Notebook 09 selected '{selected_model_name}', but Notebook 10 "
    "is configured to train the final XGBoost candidate. "
    "Re-run the corrected Notebook 09 if this assertion fails."
)

print("Selection artifact:", selection_path.name)
print("Models compared:", sorted(models_compared))
print("Selection metric:", selection_metric)
print("Selection split:", selection_split)
print("Selected model:", selected_model_name)
print("Selection verification: PASS")


Selection artifact: all_four_model_selection_info.json
Models compared: ['LightGBM', 'Logistic Regression', 'Random Forest', 'XGBoost']
Selection metric: F1
Selection split: validation
Selected model: XGBoost
Selection verification: PASS


## Cell 6 — Define the selected XGBoost model

This is the baseline XGBoost configuration used during Notebook 09 model selection.

Notebook 11 will tune the hyperparameters.


In [5]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Model:", selected_model_name)
print("XGBoost configuration created: PASS")


Model: XGBoost
XGBoost configuration created: PASS


## Cell 7 — Train XGBoost on training data only

In [6]:
training_start = time.perf_counter()

# IMPORTANT:
# Only X_train and y_train are used for fitting.
xgb_model.fit(X_train, y_train)

training_seconds = time.perf_counter() - training_start

trained_model = xgb_model

print("Selected model trained:", selected_model_name)
print("Training rows:", X_train.shape[0])
print("Training features:", X_train.shape[1])
print("Training time (seconds):", round(training_seconds, 4))


Selected model trained: XGBoost
Training rows: 3500
Training features: 590
Training time (seconds): 1.2302


## Cell 8 — Validation prediction sanity check

Validation predictions are generated only to confirm that the trained model produces valid binary predictions.

No model-selection metric is calculated in this notebook.


In [7]:
validation_predictions = trained_model.predict(X_validation)

assert len(validation_predictions) == 750
assert set(np.unique(validation_predictions)).issubset({0, 1})

print("Validation predictions generated:", len(validation_predictions))
print("Validation sanity check: PASS")


Validation predictions generated: 750
Validation sanity check: PASS


## Cell 9 — Explicit test-set protection

In [8]:
test_prediction_generated = False
test_used_for_training = False
test_used_for_selection = False

assert test_prediction_generated is False
assert test_used_for_training is False
assert test_used_for_selection is False

print("Test rows:", X_test.shape[0])
print("Test prediction generated: NO")
print("Test used for training: NO")
print("Test used for model selection: NO")


Test rows: 750
Test prediction generated: NO
Test used for training: NO
Test used for model selection: NO


## Cell 10 — Save the trained XGBoost candidate

In [9]:
trained_model_path = MODEL_DIR / "trained_candidate_model.joblib"

joblib.dump(trained_model, trained_model_path)

assert trained_model_path.exists()

print("Trained model path:", trained_model_path)
print("Trained model exists:", trained_model_path.exists())


Trained model path: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\trained_candidate_model.joblib
Trained model exists: True


## Cell 11 — Save training metadata

In [10]:
metadata_path = MODEL_DIR / "training_metadata.json"

training_metadata = {
    "selected_model": "XGBoost",
    "selection_metric": "F1",
    "selection_split": "validation",
    "selection_artifact": "all_four_model_selection_info.json",
    "models_compared": sorted(models_compared),
    "training_rows": int(X_train.shape[0]),
    "training_features": int(X_train.shape[1]),
    "validation_rows": int(X_validation.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "test_used_for_training": False,
    "test_used_for_selection": False,
    "test_prediction_generated": False,
    "training_seconds": float(training_seconds)
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2)

assert metadata_path.exists()

print("Metadata path:", metadata_path)
print("Metadata exists:", metadata_path.exists())


Metadata path: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\training_metadata.json
Metadata exists: True


## Cell 12 — Reload the saved XGBoost model and verify it

In [11]:
reloaded_model = joblib.load(trained_model_path)

assert isinstance(reloaded_model, XGBClassifier)

reloaded_validation_predictions = reloaded_model.predict(X_validation)

assert len(reloaded_validation_predictions) == 750
assert np.array_equal(
    validation_predictions,
    reloaded_validation_predictions
)

print("Reloaded model type:", type(reloaded_model).__name__)
print("Reloaded model prediction check: PASS")
print("Validation predictions after reload:", len(reloaded_validation_predictions))


Reloaded model type: XGBClassifier
Reloaded model prediction check: PASS
Validation predictions after reload: 750


## Cell 13 — Verify training metadata

In [12]:
with open(metadata_path, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

assert saved_metadata["selected_model"] == "XGBoost"
assert saved_metadata["selection_metric"] == "F1"
assert saved_metadata["selection_split"] == "validation"
assert saved_metadata["selection_artifact"] == "all_four_model_selection_info.json"

assert saved_metadata["training_rows"] == 3500
assert saved_metadata["training_features"] == 590
assert saved_metadata["validation_rows"] == 750
assert saved_metadata["test_rows"] == 750

assert saved_metadata["test_used_for_training"] is False
assert saved_metadata["test_used_for_selection"] is False
assert saved_metadata["test_prediction_generated"] is False

print("Training metadata verification: PASS")
print("Selected model:", saved_metadata["selected_model"])
print("Selection metric:", saved_metadata["selection_metric"])


Training metadata verification: PASS
Selected model: XGBoost
Selection metric: F1


## Cell 14 — Final validation

In [13]:
assert selected_model_name == "XGBoost"
assert selection_info["selected_model"] == "XGBoost"

assert selection_metric == "F1"
assert selection_split == "validation"

assert trained_model_path.exists()
assert metadata_path.exists()

assert isinstance(trained_model, XGBClassifier)

assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert len(validation_predictions) == 750
assert set(np.unique(validation_predictions)).issubset({0, 1})

assert training_metadata["selected_model"] == "XGBoost"
assert training_metadata["test_used_for_training"] is False
assert training_metadata["test_used_for_selection"] is False
assert training_metadata["test_prediction_generated"] is False

print("Selected model:", selected_model_name)
print("Model type:", type(trained_model).__name__)
print("Training rows:", X_train.shape[0])
print("Training features:", X_train.shape[1])
print("Validation rows:", X_validation.shape[0])
print("Test rows:", X_test.shape[0])
print("Test used for training: False")
print("Test used for selection: False")
print("Test prediction generated: NO")
print("Trained model exists:", trained_model_path.exists())
print("Metadata exists:", metadata_path.exists())

print("\nNotebook 10 final validation: PASS")


Selected model: XGBoost
Model type: XGBClassifier
Training rows: 3500
Training features: 590
Validation rows: 750
Test rows: 750
Test used for training: False
Test used for selection: False
Test prediction generated: NO
Trained model exists: True
Metadata exists: True

Notebook 10 final validation: PASS
